# Metro RAG System

## 1.0 - Data preprocessing

#### 1.1 -  Load Data

In [52]:
import pandas as pd
import numpy as np

df = pd.read_csv('metro-stations-in-riyadh-by-metro-line-and-station-type-2024.json')

df

,"[{""index"": 40","""metro_station_cd"": ""S02""","""metro_station_desc_en"": ""Dr Sulaiman Al Habib""","""metro_station_desc_ar"": ""\u062f.\u0633\u0644\u064a\u0645\u0627\u0646 \u0627\u0644\u062d\u0628\u064a\u0628""","""metro_line_cd"": ""Line1""","""metro_line_desc_en"": ""Blue line""","""metro_line_desc_ar"": ""\u0627\u0644\u0645\u0633\u0627\u0631 \u0627\u0644\u0623\u0632\u0631\u0642""","""metro_station_type_cd"": 1","""metro_station_type_desc_en"": ""Elevated""","""metro_station_type_desc_ar"": ""\u0645\u0631\u062a\u0641\u0639""",...,"""metro_station_seq"": 90","""comments_en"": null.83","""comments_ar"": null.83","""geoshape"": {""type"": ""Feature"".93","""geometry"": {""coordinates"": [46.77657649147416",24.77632719132187],"""type"": ""Point""}.93","""properties"": {}}.93","""geo_point_2d"": {""lon"": 46.77657649147416","""lat"": 24.77632719132187}}]"


#### 1.2 - Flatten the data (since it's 0 x 1890 it's hard for pandas to deal with that shape)

In [53]:
import json
import pandas as pd

# 1. Load the raw JSON file manually
file_path = 'metro-stations-in-riyadh-by-metro-line-and-station-type-2024.json' # <-- Change this to your actual file name
with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Check the structure to find the real rows
if isinstance(raw_data, dict):
    print("This JSON is a dictionary. The main keys are:", raw_data.keys())
        
elif isinstance(raw_data, list):
    print("This JSON is a list. Flattening it now...")
    # json_normalize flattens nested JSON automatically
    df = pd.json_normalize(raw_data)
    print(f"Success! New shape: {df.shape}")
    print(df.head(2))

This JSON is a list. Flattening it now...
Success! New shape: (94, 18)
   index metro_station_cd metro_station_desc_en metro_station_desc_ar  \
0     40              S02  Dr Sulaiman Al Habib       د.سليمان الحبيب   
1     49              S03                  KAFD         المركز المالي   

  metro_line_cd metro_line_desc_en metro_line_desc_ar  metro_station_type_cd  \
0         Line1          Blue line      المسار الأزرق                      1   
1         Line1          Blue line      المسار الأزرق                      1   

  metro_station_type_desc_en metro_station_type_desc_ar  metro_station_seq  \
0                   Elevated                      مرتفع                  2   
1                   Elevated                      مرتفع                  3   

  comments_en comments_ar geoshape.type  \
0        None        None       Feature   
1        None        None       Feature   

            geoshape.geometry.coordinates geoshape.geometry.type  \
0  [46.62654386725997, 24.811553793

In [54]:
print(df.shape)
df.head()

(94, 18)


,index,metro_station_cd,metro_station_desc_en,metro_station_desc_ar,metro_line_cd,metro_line_desc_en,metro_line_desc_ar,metro_station_type_cd,metro_station_type_desc_en,metro_station_type_desc_ar,metro_station_seq,comments_en,comments_ar,geoshape.type,geoshape.geometry.coordinates,geoshape.geometry.type,geo_point_2d.lon,geo_point_2d.lat
0,40,S02,Dr Sulaiman Al Habib,د.سليمان الحبيب,Line1,Blue line,المسار الأزرق,1,Elevated,مرتفع,2,None,None,Feature,"[46.62654386725997, 24.81155379383345]",Point,46.626544,24.811554
1,49,S03,KAFD,المركز المالي,Line1,Blue line,المسار الأزرق,1,Elevated,مرتفع,3,None,None,Feature,"[46.64364447641301, 24.76800392512061]",Point,46.643644,24.768004
2,18,S04,Al Murooj,المروج,Line1,Blue line,المسار الأزرق,4,Deep Underground,تحت الأرض على عمق كبير,4,None,None,Feature,"[46.65422135694412, 24.75474607607529]",Point,46.654221,24.754746
3,56,S05,King Fahad District,حي الملك فهد,Line1,Blue line,المسار الأزرق,4,Deep Underground,تحت الأرض على عمق كبير,5,None,None,Feature,"[46.65932671648558, 24.74491442205136]",Point,46.659327,24.744914
4,24,S08,Al Wurud 2,الورود 2,Line1,Blue line,المسار الأزرق,4,Deep Underground,تحت الأرض على عمق كبير,8,None,None,Feature,"[46.67112888545265, 24.72172740595173]",Point,46.671129,24.721727


#### 1.3 -  Function that transform the structured data into chunks of texts

In [55]:
import pandas as pd
import numpy as np

# 1. Sort the data by metro line and then by station sequence
# This is crucial so we can correctly infer neighboring stations
df = df.sort_values(by=['metro_line_cd', 'metro_station_seq']).reset_index(drop=True)

# 2. Determine previous and next stations using Pandas groupby and shift
# This creates new columns for the neighbors within the same metro line
df['prev_station_en'] = df.groupby('metro_line_cd')['metro_station_desc_en'].shift(1)
df['next_station_en'] = df.groupby('metro_line_cd')['metro_station_desc_en'].shift(-1)

df['prev_station_ar'] = df.groupby('metro_line_cd')['metro_station_desc_ar'].shift(1)
df['next_station_ar'] = df.groupby('metro_line_cd')['metro_station_desc_ar'].shift(-1)

# 3. Dense Chunking Function
def textify_metro(row):
    # Using a direct, dense key-value style to eliminate token bloat
    chunk = (
        f"Station: {row['metro_station_desc_en']} ({row['metro_station_desc_ar']}). "
        f"Line: {row['metro_line_desc_en']} ({row['metro_line_cd']}). "
        f"Type: {row['metro_station_type_desc_en']}. "
        f"Sequence: {row['metro_station_seq']}. "
    )
    
    # Add relational context if neighbors exist (ignores ends of the line)
    if pd.notna(row['prev_station_en']):
        chunk += f"Previous station: {row['prev_station_en']} ({row['prev_station_ar']}). "
    if pd.notna(row['next_station_en']):
        chunk += f"Next station: {row['next_station_en']} ({row['next_station_ar']})."
        
    return str(chunk).strip()

#### 1.4 - Applying the function

In [56]:
df['text_chunk'] = [textify_metro(row) for index, row in df.iterrows()]

# Print the result to see the difference
print("Dense text chunk for a station in the middle of a line:\n")
print(df['text_chunk'].iloc[10])

Dense text chunk for a station in the middle of a line:

Station: Bank Albilad (بنك البلاد). Line: Blue line (Line1). Type: Deep Underground. Sequence: 11. Previous station: Alinma Bank (مصرف الانماء). Next station: King Fahad Library (مكتبة الملك فهد).


#### 1.5 - Importing the embedding model (we need a model that support both Arabic and English)

In [57]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('intfloat/multilingual-e5-small')

#### 1.6 - Embedding the data

In [58]:
sentences = df['text_chunk'].tolist()
embeddings = embedding_model.encode(sentences, show_progress_bar=True)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

#### 1.7 - Storing the embeddings vectors

In [59]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print(f"Successfully added {index.ntotal} vectors to the FAISS database!")

Successfully added 94 vectors to the FAISS database!


#### 1.8 - The function to get the most similarity from the vectors database

In [60]:
import numpy as np

def retrieve_relevant_chunks(query, embedding_model, index, df, k=5):
    """
    Takes a natural language query, embeds it, and retrieves the top-k 
    most relevant chunks from the vector database.
    
    Parameters:
    - query (str): The user's search question.
    - embedding_model: The SentenceTransformer embedding model.
    - index: The FAISS vector database index.
    - df (pd.DataFrame): The dataframe containing the 'text_chunk' column.
    - k (int): The number of results to return (default is 3).
    
    Returns:
    - list of dicts: Contains the rank, distance (score), and text of the retrieved chunks.
    """
    # 1. Convert the text query into a numerical vector
    query_vector = embedding_model.encode([query])
    
    # 2. Search the FAISS index for the top-k closest vectors
    distances, indices = index.search(np.array(query_vector), k)
    
    # 3. Compile the results
    retrieved_chunks = []
    
    # indices[0] contains the row numbers, distances[0] contains the similarity scores
    for rank, (idx, distance) in enumerate(zip(indices[0], distances[0])):
        chunk_text = df['text_chunk'].iloc[idx]
        
        retrieved_chunks.append({
            "rank": rank + 1,
            "row_index": idx,
            "distance": round(float(distance), 4), # Lower distance means higher relevance
            "text": chunk_text
        })
        
    return retrieved_chunks

## 2.0 - Setup needed functions and tools

#### 2.1 - System prompt for the output

In [101]:
def build_prompt(question, retrieved_docs):
    context = "\n\n".join(retrieved_docs)
    return f"""You are answering a user query based ONLY on the provided context.

CRUCIAL RULES:
1. Identify the language of the Question (e.g., Arabic or English). You MUST answer in that exact same language.
2. If the answer is not found in the context, reply politely in the user's language that the information is not available in the provided data. Do not guess.
3. Use Markdown formatting (bolding, bullet points) to make your answer structured and easy to read.

Context: 
{context}

Question: {question}"""

#### 2.2 - Setup the LLM API

In [102]:
import boto3

client = boto3.client(
    "bedrock-runtime",
    region_name='us-east-1',
)

model_id = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

def add_user_message(messages, text):
    user_message = {"role": "user", "content": [{"text": text}]}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": [{"text": text}]}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)

    return response["output"]["message"]["content"][0]["text"]

/Users/saleh/Documents/rag_sys/metro_rag_system/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


#### 2.3 - Function that retrieve the LLM model answer

In [108]:
def ask_metro_rag(user_question):
    # Extracting just the text strings from the results
    raw_results = retrieve_relevant_chunks(user_question, embedding_model, index, df, k=3)
    retrieved_docs = [res['text'] for res in raw_results]
    
    # 2. Build the final prompt using your exact function
    final_prompt = build_prompt(user_question, retrieved_docs)
    
    # 3. Initialize the messages list and add the user's prompt
    messages = []
    add_user_message(messages, final_prompt)
    
    # 4. Define the System Role
    system_role = (
        "You are a highly capable, bilingual (Arabic/English) AI assistant for the Riyadh Metro system. "
        "You are precise, factual, and strictly rely on provided data. You excel at presenting information "
        "clearly using Markdown formatting."
    )    
    # 5. Call Bedrock chat function
    answer = chat(
        messages=messages, 
        system=system_role, 
        temperature=0.1 
    )
    
    return answer

## 3.0 - Evaluation

#### 3.1 - Evaluation Questions

In [104]:
test_queries = [
    "What type of station is Dr Sulaiman Al Habib, and which line is it on?", # Fact Retrieval
    "What is the station immediately following KAFD on the Blue Line?",       # Relational Logic
    "How many elevated stations are there in total on the Blue Line?"         # Aggregation (Stress Test)
]

print("SYSTEM EVALUATION (WITH LLM GENERATION)\n" + "="*40)
for i, query in enumerate(test_queries, 1):
    print(f"\nTest Q{i}: '{query}'")
    
    # 1. Print the raw retrieval data so you can check retrieval quality
    raw_results = retrieve_relevant_chunks(query, embedding_model, index, df, k=3)
    print("  [Retrieved Contexts]:")
    for res in raw_results:
        print(f"    -> [Rank {res['rank']} | Score: {res['distance']}] {res['text']}")
    
    # 2. Call your actual RAG function to hit the Bedrock API
    # Using the glue function we created in the previous step
    print("  [LLM Response]:")
    llm_response = ask_metro_rag(query)
    print(f"    🤖 Claude: {llm_response}")
    print("-" * 50)

SYSTEM EVALUATION (WITH LLM GENERATION)

Test Q1: 'What type of station is Dr Sulaiman Al Habib, and which line is it on?'
  [Retrieved Contexts]:
    -> [Rank 1 | Score: 0.1757] Station: Dr Sulaiman Al Habib (د.سليمان الحبيب). Line: Blue line (Line1). Type: Elevated. Sequence: 2. Previous station: SAB (الأول). Next station: KAFD (المركز المالي).
    -> [Rank 2 | Score: 0.188] Station: SAB (الأول). Line: Blue line (Line1). Type: Elevated. Sequence: 1. Next station: Dr Sulaiman Al Habib (د.سليمان الحبيب).
    -> [Rank 3 | Score: 0.2465] Station: KAFD (المركز المالي). Line: Blue line (Line1). Type: Elevated. Sequence: 3. Previous station: Dr Sulaiman Al Habib (د.سليمان الحبيب). Next station: Al Murooj (المروج).
  [LLM Response]:
    🤖 Claude: # Dr Sulaiman Al Habib Station

Based on the provided information:

- **Type:** Elevated
- **Line:** Blue Line (Line 1)

This station is the second station in the sequence on the Blue Line, located between SAB (الأول) and KAFD (المركز المالي).
-----

## 4.0 - Frontend part

#### 4.1 - UI Wrapper function

In [ ]:
import gradio as gr

def answer_question_with_ui(user_question):
    """
    This function acts as the bridge between Gradio and your RAG engine.
    It returns TWO things: the LLM's answer, and the raw retrieved context.
    """
    # Step 1: Retrieve the data using your existing function
    raw_results = retrieve_relevant_chunks(user_question, embedding_model, index, df, k=3)
    retrieved_docs = [res['text'] for res in raw_results]
    
    # Format the retrieved context so it looks nice in the UI box
    formatted_context = ""
    for res in raw_results:
        formatted_context += f"[Rank {res['rank']} | Score: {res['distance']}]\n{res['text']}\n\n"
        
    # Step 2: Build the prompt (using your function)
    final_prompt = build_prompt(user_question, retrieved_docs)
    
    # Step 3: Call the Bedrock LLM (using your function)
    messages = []
    add_user_message(messages, final_prompt)
    system_role = "You are a precise, factual AI assistant for the Riyadh Metro system."
    
    try:
        # Temperature 0.1 for strict, factual answers
        llm_response = chat(messages=messages, system=system_role, temperature=0.1)
    except Exception as e:
        # Just in case AWS credentials expire or timeout
        llm_response = f"⚠️ Connection Error: {str(e)}"
        
    # Return both items so Gradio can display them in two separate output boxes
    return llm_response, formatted_context.strip()

#### 4.2 - Build the Gradio Interface

In [143]:
demo = gr.Interface(
    fn=answer_question_with_ui,
    
    # The Input UI
    inputs=gr.Textbox(
        label="Ask about the Riyadh Metro:", 
        placeholder="e.g., Which line is KAFD station on?",
        lines=2
    ),
    
    # The Output UI (Two boxes: one for the AI, one for Transparency)
    outputs=[
        gr.Markdown(label="Answer"),
        gr.Textbox(label="Retrieved Source Data", lines=6)
    ],
    
    # The Title and Description
    title="🚇 Riyadh Metro RAG Assistant",
    description=(
        "Ask any question about the Riyadh Metro system. "
        "The system will retrieve the exact tabular data from the FAISS vector database, "
        "and the LLM will generate a highly factual answer based **only** on that data."
    ),
    
    # Pre-loaded Example Questions (Clicking these auto-fills the input box)
    examples=[
        ["What type of station is Dr Sulaiman Al Habib, and which line is it on?"],
        ["What is the station immediately following KAFD on the Blue Line?"],
        ["How many elevated stations are there in total on the Blue Line?"]
    ],
    
    # Styling theme
    theme=gr.themes.Soft()
)

#### 4.3 - Launch it

In [142]:
demo.launch()

Running on local URL:  http://127.0.0.1:7887


/Users/saleh/Documents/rag_sys/metro_rag_system/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)



To create a public link, set `share=True` in `launch()`.
